# JarCut GPU — очистка звука + OmniVoice

Этот ноутбук — GPU-сервер для JarCut. Он умеет две вещи:
- **очистка звука**: голос отдельно, фон отдельно, из голоса убираются эхо и шум;
- **озвучка** клоном голоса через OmniVoice.

### Как запустить
1. **Среда выполнения → Сменить тип среды → T4 GPU → Сохранить.**
2. **Среда выполнения → Выполнить всё** (Ctrl+F9).
3. Подожди. Первый запуск дольше: скачиваются модели (5–10 минут).
4. В конце появится строка `JARCUT_GPU_URL=...` — скопируй её в файл `.env` рядом с `server.py` (старую строку `JARCUT_GPU_URL` замени). Перезапускать `server.py` не нужно.

### Важно
- Не закрывай эту вкладку, пока работаешь. Если Colab отключился — снова **Выполнить всё** и новую строку в `.env`.
- Красные предупреждения pip про конфликты версий при установке — это нормально.
- Если что-то упало — запусти ячейку **4. Лог** и пришли её вывод в чат.

In [ ]:
#@title 1. Установка (3–6 минут)
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    print("⚠️ GPU НЕ ПОДКЛЮЧЁН! Среда выполнения → Сменить тип среды → T4 GPU, потом «Выполнить всё» заново.")
else:
    print("GPU:", gpu.stdout.strip())

!pip install -q omnivoice "audio-separator[gpu]" fastapi "uvicorn[standard]" python-multipart soundfile
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
print("\n✅ Установка завершена")

## 2. Код сервера
Эта ячейка просто записывает файл `jarcut_gpu.py` — менять ничего не нужно.

In [ ]:
%%writefile /content/jarcut_gpu.py
"""JarCut GPU-сервер — запускается внутри Google Colab (ноутбук JarCut_GPU.ipynb).

Что умеет:
  * очистка звука: голос отдельно, фон отдельно (audio-separator, Roformer),
    плюс удаление эха и шума из голоса;
  * озвучка через OmniVoice (Python API напрямую, без Gradio-демо):
    голос спикера кодируется один раз и переиспользуется.

Все запросы требуют ключ: заголовок X-Jarcut-Key или ?key=... в адресе.

  GET    /health
  POST   /separate              multipart: file, model, dereverb, denoise, aggressive -> {job_id}
  GET    /jobs/{job_id}         статус задачи очистки
  GET    /files/{job_id}/{name} name: vocals_raw | vocals_clean | background
  DELETE /jobs/{job_id}
  POST   /voices                multipart: file, key, ref_text -> голос запомнен под ключом
  POST   /tts                   json: text, language, voice_key, настройки -> audio/wav
"""

import gc
import io
import os
import re
import shutil
import threading
import time
import traceback
import uuid
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf
from fastapi import Body, FastAPI, File, Form, Request, UploadFile
from fastapi.responses import FileResponse, JSONResponse, Response

VERSION = "1"
ACCESS_KEY = os.environ.get("JARCUT_KEY", "").strip()
WORK_DIR = Path(os.environ.get("JARCUT_WORK_DIR", "/content/jarcut_work"))
MODELS_DIR = Path(os.environ.get("JARCUT_MODELS_DIR", "/content/jarcut_models"))
OMNIVOICE_MODEL = os.environ.get("OMNIVOICE_MODEL", "k2-fsa/OmniVoice")
PORT = int(os.environ.get("JARCUT_PORT", "8000"))

# Модели audio-separator (файлы скачиваются сами при первом запуске).
SEPARATION_MODELS = {
    "kim": "vocals_mel_band_roformer.ckpt",                      # MelBand Roformer (Kimberley Jensen)
    "viperx": "model_bs_roformer_ep_317_sdr_12.9755.ckpt",       # BS-Roformer (ViperX)
}
DEREVERB_MODEL = "dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt"
DENOISE_MODEL = "denoise_mel_band_roformer_aufr33_sdr_27.9959.ckpt"
DENOISE_AGGRESSIVE_MODEL = "denoise_mel_band_roformer_aufr33_aggr_sdr_27.9768.ckpt"
RESULT_NAMES = ("vocals_raw", "vocals_clean", "background")
MAX_VOICES = 600

WORK_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title="JarCut GPU")

# Одна видеокарта: очистка и озвучка никогда не идут одновременно, иначе
# бесплатной T4 может не хватить памяти.
gpu_lock = threading.Lock()

state: dict[str, Any] = {"tts": "loading", "tts_error": "", "started": time.time()}
tts_model: Any = None
voices: dict[str, Any] = {}
jobs: dict[str, dict[str, Any]] = {}


def json_error(status: int, code: str, message: str) -> JSONResponse:
    return JSONResponse({"ok": False, "code": code, "error": message}, status_code=status)


def parse_bool(value: Any, default: bool = False) -> bool:
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in ("1", "true", "yes", "on")


@app.middleware("http")
async def require_key(request: Request, call_next):
    if ACCESS_KEY:
        supplied = request.headers.get("x-jarcut-key") or request.query_params.get("key") or ""
        if supplied != ACCESS_KEY:
            return json_error(401, "BAD_KEY", "Неверный ключ доступа")
    return await call_next(request)


def free_gpu_memory() -> None:
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def gpu_info() -> dict[str, Any]:
    try:
        import torch

        if not torch.cuda.is_available():
            return {"name": "нет GPU (CPU)", "totalGb": 0, "freeGb": 0}
        free, total = torch.cuda.mem_get_info()
        return {
            "name": torch.cuda.get_device_name(0),
            "totalGb": round(total / 1024 ** 3, 1),
            "freeGb": round(free / 1024 ** 3, 1),
        }
    except Exception as exc:
        return {"name": f"неизвестно ({exc})", "totalGb": 0, "freeGb": 0}


@app.get("/health")
def health() -> dict[str, Any]:
    return {
        "ok": True,
        "version": VERSION,
        "tts": state["tts"],
        "ttsError": state["tts_error"],
        "gpu": gpu_info(),
        "voices": len(voices),
        "jobs": {job_id: job["status"] for job_id, job in jobs.items()},
        "uptime": round(time.time() - state["started"]),
    }


# ---------------------------------------------------------------------------
# OmniVoice
# ---------------------------------------------------------------------------


def load_tts() -> None:
    """Загружает OmniVoice один раз при старте (в фоне — /health уже отвечает)."""
    global tts_model
    try:
        import torch
        from omnivoice import OmniVoice

        device = "cuda:0" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        print(f"[tts] Загружаю {OMNIVOICE_MODEL} на {device} ...", flush=True)
        tts_model = OmniVoice.from_pretrained(OMNIVOICE_MODEL, device_map=device, dtype=dtype)
        state["tts"] = "ready"
        print("[tts] OmniVoice готов", flush=True)
    except Exception as exc:
        state["tts"] = "error"
        state["tts_error"] = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()


def make_voice_prompt(path: Path, ref_text: str) -> Any:
    try:
        return tts_model.create_voice_clone_prompt(ref_audio=str(path), ref_text=ref_text or None)
    except ValueError as exc:
        # "Reference audio is empty after silence removal" — очень тихий или
        # очень короткий референс. Пробуем без предобработки.
        if "silence" not in str(exc).lower():
            raise
        return tts_model.create_voice_clone_prompt(ref_audio=str(path), ref_text=ref_text or None, preprocess_prompt=False)


@app.post("/voices")
def register_voice(file: UploadFile = File(...), key: str = Form(...), ref_text: str = Form("")) -> Any:
    if state["tts"] != "ready":
        return json_error(503, "TTS_NOT_READY", f"OmniVoice ещё не готов ({state['tts']}) {state['tts_error']}".strip())
    if not re.fullmatch(r"[0-9a-fA-F]{8,64}", key or ""):
        return json_error(400, "BAD_VOICE_KEY", "Ключ голоса должен быть hex-строкой")

    voices_dir = WORK_DIR / "voices"
    voices_dir.mkdir(exist_ok=True)
    path = voices_dir / f"{key}.wav"
    with open(path, "wb") as handle:
        shutil.copyfileobj(file.file, handle)

    try:
        with gpu_lock:
            prompt = make_voice_prompt(path, ref_text.strip())
    except Exception as exc:
        traceback.print_exc()
        return json_error(500, "VOICE_FAILED", f"{type(exc).__name__}: {exc}")
    finally:
        path.unlink(missing_ok=True)
        free_gpu_memory()

    voices[key] = prompt
    while len(voices) > MAX_VOICES:
        voices.pop(next(iter(voices)))
    return {"ok": True, "key": key, "refText": getattr(prompt, "ref_text", "")}


@app.post("/tts")
def tts(body: dict = Body(...)) -> Any:
    if state["tts"] != "ready":
        return json_error(503, "TTS_NOT_READY", f"OmniVoice ещё не готов ({state['tts']}) {state['tts_error']}".strip())

    text = str(body.get("text") or "").strip()
    if not text:
        return json_error(400, "EMPTY_TEXT", "Пустой текст")
    prompt = voices.get(str(body.get("voice_key") or ""))
    if prompt is None:
        return json_error(404, "VOICE_NOT_FOUND", "Голос не найден (Colab перезапускали?) — зарегистрируйте заново")

    try:
        import torch
        from omnivoice import OmniVoiceGenerationConfig

        config = OmniVoiceGenerationConfig(
            num_step=int(body.get("num_step") or 32),
            guidance_scale=float(body.get("guidance_scale") or 2.0),
            denoise=parse_bool(body.get("denoise"), True),
            postprocess_output=parse_bool(body.get("postprocess_output"), True),
            pad_duration=float(body.get("pad_duration", 0.1)),
            fade_duration=float(body.get("fade_duration", 0.1)),
        )
        kwargs: dict[str, Any] = {
            "text": text,
            "language": (str(body.get("language") or "").strip() or None),
            "voice_clone_prompt": prompt,
            "generation_config": config,
        }
        duration = float(body.get("duration") or 0)
        speed = float(body.get("speed") or 1.0)
        if duration > 0:
            kwargs["duration"] = duration
        elif abs(speed - 1.0) > 1e-3:
            kwargs["speed"] = speed
        instruct = str(body.get("instruct") or "").strip()
        if instruct:
            kwargs["instruct"] = instruct

        with gpu_lock, torch.no_grad():
            try:
                audio = tts_model.generate(**kwargs)
            except Exception as exc:
                # Незнакомое модели название языка — пробуем без языка.
                if kwargs.get("language") and "lang" in str(exc).lower():
                    print(f"[tts] язык {kwargs['language']!r} не принят ({exc}), генерирую без языка", flush=True)
                    kwargs["language"] = None
                    audio = tts_model.generate(**kwargs)
                else:
                    raise
    except Exception as exc:
        traceback.print_exc()
        return json_error(500, "TTS_FAILED", f"{type(exc).__name__}: {exc}")
    finally:
        free_gpu_memory()

    waveform = np.asarray(audio[0], dtype=np.float32).reshape(-1)
    buffer = io.BytesIO()
    sf.write(buffer, waveform, tts_model.sampling_rate, format="WAV", subtype="PCM_16")
    return Response(
        buffer.getvalue(),
        media_type="audio/wav",
        headers={"X-Duration": f"{waveform.size / tts_model.sampling_rate:.3f}"},
    )


# ---------------------------------------------------------------------------
# Очистка звука (audio-separator)
# ---------------------------------------------------------------------------


def run_separator(model_filename: str, input_path: Path, out_dir: Path, names: dict[str, str]) -> None:
    """Один прогон одной модели. names: имя дорожки модели -> имя файла."""
    from audio_separator.separator import Separator

    separator = Separator(output_dir=str(out_dir), output_format="FLAC", model_file_dir=str(MODELS_DIR))
    try:
        separator.load_model(model_filename=model_filename)
        separator.separate(str(input_path), names)
    finally:
        del separator
        free_gpu_memory()


def find_output(out_dir: Path, name: str) -> Path:
    exact = out_dir / f"{name}.flac"
    if exact.exists():
        return exact
    matches = sorted(out_dir.glob(f"{name}.*"))
    if matches:
        return matches[0]
    present = ", ".join(sorted(path.name for path in out_dir.iterdir()))
    raise RuntimeError(f"Модель не создала файл {name} (в папке: {present})")


def set_step(job: dict[str, Any], step: str) -> None:
    job["step"] = step
    print(f"[separate {job['id']}] {step}", flush=True)


def separation_pipeline(job: dict[str, Any]) -> None:
    out_dir: Path = job["dir"]
    options = job["options"]

    set_step(job, "разделение: голос / фон")
    run_separator(
        SEPARATION_MODELS[options["model"]], job["input"], out_dir,
        # У Kim дорожки называются vocals/other, у ViperX — vocals/instrumental.
        {"vocals": "vocals_raw", "other": "background", "instrumental": "background"},
    )
    current = find_output(out_dir, "vocals_raw")
    find_output(out_dir, "background")

    if options["dereverb"]:
        set_step(job, "удаление эха из голоса")
        run_separator(DEREVERB_MODEL, current, out_dir, {"noreverb": "vocals_dry", "reverb": "junk_reverb"})
        current = find_output(out_dir, "vocals_dry")

    if options["denoise"]:
        set_step(job, "удаление шума из голоса" + (" (агрессивно)" if options["aggressive"] else ""))
        model = DENOISE_AGGRESSIVE_MODEL if options["aggressive"] else DENOISE_MODEL
        run_separator(model, current, out_dir, {"dry": "vocals_denoised", "other": "junk_noise"})
        current = find_output(out_dir, "vocals_denoised")

    final = out_dir / "vocals_clean.flac"
    if current.resolve() != final.resolve():
        shutil.copyfile(current, final)
    for junk in list(out_dir.glob("junk_*")) + [job["input"]]:
        Path(junk).unlink(missing_ok=True)


def separation_worker(job_id: str) -> None:
    job = jobs[job_id]
    try:
        job["step"] = "ждёт видеокарту"
        with gpu_lock:
            job["status"] = "running"
            job["started"] = time.time()
            separation_pipeline(job)
        job["status"] = "done"
        set_step(job, "готово")
    except Exception as exc:
        traceback.print_exc()
        job["status"] = "error"
        job["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        job["finished"] = time.time()


@app.post("/separate")
def separate(
    file: UploadFile = File(...),
    model: str = Form("kim"),
    dereverb: str = Form("1"),
    denoise: str = Form("1"),
    aggressive: str = Form("0"),
) -> Any:
    job_id = uuid.uuid4().hex[:12]
    job_dir = WORK_DIR / "jobs" / job_id
    job_dir.mkdir(parents=True, exist_ok=True)
    suffix = Path(file.filename or "audio.flac").suffix.lower() or ".flac"
    input_path = job_dir / f"input{suffix}"
    with open(input_path, "wb") as handle:
        shutil.copyfileobj(file.file, handle)

    model = (model or "kim").strip().lower()
    jobs[job_id] = {
        "id": job_id,
        "dir": job_dir,
        "input": input_path,
        "options": {
            "model": model if model in SEPARATION_MODELS else "kim",
            "dereverb": parse_bool(dereverb, True),
            "denoise": parse_bool(denoise, True),
            "aggressive": parse_bool(aggressive, False),
        },
        "status": "queued",
        "step": "в очереди",
        "error": "",
        "created": time.time(),
        "started": None,
        "finished": None,
    }
    threading.Thread(target=separation_worker, args=(job_id,), daemon=True).start()
    return {"ok": True, "job_id": job_id}


@app.get("/jobs/{job_id}")
def job_status(job_id: str) -> Any:
    job = jobs.get(job_id)
    if job is None:
        return json_error(404, "JOB_NOT_FOUND", "Задача не найдена (Colab перезапускали?)")
    started = job["started"] or job["created"]
    elapsed = (job["finished"] or time.time()) - started
    files = [name for name in RESULT_NAMES if (job["dir"] / f"{name}.flac").exists()] if job["status"] == "done" else []
    return {
        "ok": True,
        "status": job["status"],
        "step": job["step"],
        "error": job["error"],
        "elapsed": round(elapsed, 1),
        "files": files,
    }


@app.get("/files/{job_id}/{name}")
def job_file(job_id: str, name: str) -> Any:
    job = jobs.get(job_id)
    if job is None:
        return json_error(404, "JOB_NOT_FOUND", "Задача не найдена")
    if name not in RESULT_NAMES:
        return json_error(400, "BAD_NAME", f"Нет такого файла: {name}")
    path = job["dir"] / f"{name}.flac"
    if not path.exists():
        return json_error(404, "FILE_NOT_READY", f"Файл {name} ещё не готов")
    return FileResponse(str(path), media_type="audio/flac", filename=f"{name}.flac")


@app.delete("/jobs/{job_id}")
def delete_job(job_id: str) -> Any:
    job = jobs.get(job_id)
    if job is None:
        return {"ok": True}
    if job["status"] in ("queued", "running"):
        return json_error(409, "JOB_RUNNING", "Задача ещё выполняется")
    shutil.rmtree(job["dir"], ignore_errors=True)
    jobs.pop(job_id, None)
    return {"ok": True}


if __name__ == "__main__":
    import uvicorn

    threading.Thread(target=load_tts, daemon=True).start()
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

In [ ]:
#@title 3. Запуск сервера и ссылка для .env
import json, os, re, secrets, subprocess, sys, time, urllib.request

KEY_FILE = "/content/jarcut_key.txt"
if os.path.exists(KEY_FILE):
    KEY = open(KEY_FILE).read().strip()
else:
    KEY = secrets.token_hex(12)
    with open(KEY_FILE, "w") as f:
        f.write(KEY)

# Если ячейку запускают повторно — останавливаем прошлый сервер и туннель.
for _name in ("server_proc", "tunnel_proc"):
    _proc = globals().get(_name)
    if _proc is not None and _proc.poll() is None:
        _proc.terminate()
        try:
            _proc.wait(timeout=10)
        except Exception:
            _proc.kill()

env = dict(os.environ, JARCUT_KEY=KEY, JARCUT_PORT="8000")
server_proc = subprocess.Popen(
    [sys.executable, "-u", "/content/jarcut_gpu.py"],
    stdout=open("/content/jarcut_server.log", "w"), stderr=subprocess.STDOUT, env=env,
)

def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:8000/health?key={KEY}", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

def log_tail(n=30):
    try:
        return "".join(open("/content/jarcut_server.log").readlines()[-n:])
    except Exception:
        return ""

print("Запускаю сервер...")
for _ in range(90):
    if health():
        break
    if server_proc.poll() is not None:
        print(log_tail())
        raise SystemExit("❌ Сервер упал при запуске — лог выше. Пришли его в чат.")
    time.sleep(2)
else:
    print(log_tail())
    raise SystemExit("❌ Сервер не запустился за 3 минуты — лог выше.")

print("Открываю туннель...")
tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if match:
        url = match.group(0)
        break
    time.sleep(1)
if not url:
    print(open("/content/cloudflared.log").read()[-2000:])
    raise SystemExit("❌ Туннель не открылся — запусти эту ячейку ещё раз.")

print("Загружаю OmniVoice (первый раз скачивается модель — несколько минут)...")
for _ in range(400):
    h = health()
    if h and h["tts"] != "loading":
        break
    time.sleep(3)
h = health() or {}
if h.get("tts") != "ready":
    print(log_tail(40))
    print("⚠️ OmniVoice не загрузился:", h.get("ttsError"), "— очистка звука при этом работает. Пришли лог в чат.")

print("\n" + "=" * 72)
print("✅ ГОТОВО. Скопируй эту строку в файл .env рядом с server.py:\n")
print(f"JARCUT_GPU_URL={url}/?key={KEY}")
print("\n(старую строку JARCUT_GPU_URL замени; server.py перезапускать не нужно)")
print("=" * 72)
print("GPU:", h.get("gpu"))

In [ ]:
#@title 4. Лог (если что-то не так — запусти и пришли вывод в чат)
print(open("/content/jarcut_server.log").read()[-8000:])
print("\n--- туннель ---")
print(open("/content/cloudflared.log").read()[-1500:])